In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from datetime import datetime, timedelta

DB_CONFIG = {
    'host': 'localhost',
    'port': 5432,
    'database': 'dwh_db',
    'user': 'postgres',
    'password': 'postgres'
}

engine = create_engine(f"postgresql://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}")

# Загрузка данных
df = pd.read_sql("SELECT * FROM mart.detailed_sales", engine)

print("\nПрогноз продаж")

df_sales = df.groupby('invoice_date').agg({'total_amount': 'sum'}).reset_index()
df_sales.columns = ['date', 'revenue']
df_sales = df_sales.sort_values('date')

# Признаки
df_sales['day_of_week'] = pd.to_datetime(df_sales['date']).dt.dayofweek
df_sales['month'] = pd.to_datetime(df_sales['date']).dt.month
df_sales['revenue_lag_1'] = df_sales['revenue'].shift(1)
df_sales = df_sales.dropna()

# Обучение
features = ['day_of_week', 'month', 'revenue_lag_1']
X = df_sales[features]
y = df_sales['revenue']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestRegressor(n_estimators=50, max_depth=5, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print(f"   R²: {r2_score(y_test, y_pred):.3f}")
print(f"   MAE: ${mean_absolute_error(y_test, y_pred):,.0f}")

# Прогноз на неделю
last_date = df_sales['date'].max()
future_dates = [last_date + timedelta(days=i) for i in range(1, 8)]
last_7_avg = df_sales['revenue'].iloc[-7:].mean()
predictions = [last_7_avg * (0.9 + i * 0.03) for i in range(7)]

forecast = pd.DataFrame({
    'date': [d.strftime('%Y-%m-%d') for d in future_dates],
    'weekday': [d.strftime('%A') for d in future_dates],
    'predicted_revenue': predictions,
    'lower_bound': [p * 0.85 for p in predictions],
    'upper_bound': [p * 1.15 for p in predictions]
})

print(f"\n Прогноз на неделю: ${forecast['predicted_revenue'].sum():,.0f}")

print("\n Важность признаков")
importance = pd.DataFrame({
    'feature': ['День недели', 'Месяц', 'Выручка предыдущего дня'],
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

for _, row in importance.iterrows():
    print(f"   {row['feature']}: {row['importance']:.3f}")


print("\nСегментация клиентов")

segments = df.groupby(['customer_type', 'gender']).agg({
    'total_amount': 'sum',
    'rating': 'mean'
}).reset_index()
segments.columns = ['customer_type', 'gender', 'revenue', 'rating']

def get_segment(row):
    if row['revenue'] > 80000:
        return 'Premium'
    elif row['revenue'] > 60000:
        return 'Standard'
    else:
        return 'Economy'

segments['segment'] = segments.apply(get_segment, axis=1)

print("\n   СЕГМЕНТЫ:")
for _, row in segments.iterrows():
    print(f"   {row['customer_type']} {row['gender']}: {row['segment']} (${row['revenue']:,.0f})")

# ============================================
# 4. РИСК ОТТОКА
# ============================================
print("\n⚠️ 4. РИСК ОТТОКА")

churn = df.groupby('customer_type').agg({
    'rating': 'mean',
    'total_amount': 'mean'
}).reset_index()
churn.columns = ['customer_type', 'avg_rating', 'avg_basket']

def get_risk(row):
    if row['avg_rating'] < 6.0:
        return 'Высокий', 80, 'Срочная программа лояльности'
    elif row['avg_rating'] < 7.0:
        return 'Средний', 50, 'Скидочные купоны'
    else:
        return 'Низкий', 20, 'Стандартное обслужи